
# Computational Complexity Analysis

This notebook evaluates the computational complexity of the proposed pulmonary nodule analysis framework. The evaluation covers the segmentation and classification components separately, followed by an assessment of the combined framework.

The analysis is organized into three main parts:

1. **Segmentation Complexity:** Evaluates the five segmentation architectures, including the 2D baseline, 2.5D baseline, encoder attention, decoder attention, and full attention models. The evaluation includes parameter count, model size, MACs, FLOPs, training memory, inference memory, latency, and throughput. All segmentation models are evaluated using the GPU.

2. **Classification Complexity:** Evaluates the XGBoost classifiers using 776 input features. The analysis includes model structure (number of trees and maximum tree depth), model size, inference memory, prediction latency, and throughput. Since classification is performed on the CPU, its computational measurements are reported separately from those of the GPU-based segmentation stage.

3. **Overall Framework Complexity:** Summarizes the computational requirements of the complete framework by combining the model storage and computational costs of the selected segmentation architecture and the XGBoost classification component. The total model size is calculated from both components. However, CPU memory usage during classification and GPU memory usage during segmentation are reported separately because the two stages operate on different hardware. The component-level measurements are not interpreted as an independently measured end-to-end execution time.

-----------

### 1. Segmentation Complexity

#### 1.1. 2D_Baseline Complexity

In [ ]:

# ============================================================
# Segmentation Computational Complexity Analysis
# Five Ablation Architectures
#
# Metrics:
# Parameters, Model Size, MACs, FLOPs,
# Training Memory, Inference Memory,
# Latency, and Throughput
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Warning: thop is not installed. MACs and FLOPs will be unavailable.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH_SIZE = 320
BATCH_SIZE_THROUGHPUT = BATCH_SIZE

N_WARMUP = 25
N_TIMED = 30

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Architecture configurations
# ------------------------------------------------------------

ARCH_CONFIGS = {
    "2D_Baseline": {
        "in_ch": 1,
        "enc_att": False,
        "dec_att": False
    },
}

# ------------------------------------------------------------
# Model utilities
# ------------------------------------------------------------

def build_profiling_model(cfg):
    return SwinUNetAdaptive_DS(
        in_ch=cfg["in_ch"],
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    )


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def model_size_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    total_bytes += sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )
    return total_bytes / (1024 ** 2)


def unpack_model_output(output, training=True):
    if isinstance(output, (tuple, list)):
        main_output = output[0]
        aux2 = output[1] if len(output) > 1 else None
        aux3 = output[2] if len(output) > 2 else None
        attn_mask = (
            output[3]
            if training and len(output) > 3
            else None
        )
    else:
        main_output = output
        aux2 = aux3 = attn_mask = None

    return main_output, aux2, aux3, attn_mask


# ------------------------------------------------------------
# MACs and FLOPs
# ------------------------------------------------------------

def measure_flops(model, in_ch):

    if not HAS_THOP:
        return np.nan, np.nan

    dummy = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    try:
        model.eval()
        macs, _ = thop_profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
        return macs, macs * 2

    except Exception as error:
        print(f"Warning: MACs/FLOPs calculation failed: {error}")
        return np.nan, np.nan

    finally:
        del dummy


# ------------------------------------------------------------
# Training memory
# ------------------------------------------------------------

def measure_train_memory(model, in_ch, loss_fn):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.train()

    x = torch.randn(
        BATCH_SIZE, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    y = torch.randint(
        0, 2,
        (BATCH_SIZE, 1, PATCH_SIZE, PATCH_SIZE),
        device=device
    ).float()

    try:
        output = model(x)

        main_output, aux2, aux3, attn_mask = (
            unpack_model_output(output, training=True)
        )

        loss = loss_fn(
            main_output,
            y,
            aux2=aux2,
            aux3=aux3,
            attn_mask=attn_mask
        )

        loss.backward()
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    finally:
        model.zero_grad(set_to_none=True)

        for variable in [
            x, y, output, main_output,
            aux2, aux3, attn_mask, loss
        ]:
            if variable is not None:
                del variable

        gc.collect()
        torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Inference memory
# ------------------------------------------------------------

def measure_inference_memory(model, in_ch):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()

    with torch.no_grad():
        x = torch.randn(
            BATCH_SIZE_THROUGHPUT,
            in_ch,
            PATCH_SIZE,
            PATCH_SIZE,
            device=device
        )

        output = model(x)
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    del x, output
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Latency and throughput
# ------------------------------------------------------------

def measure_inference_speed(model, in_ch):

    model.eval()

    x_single = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_single)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        latency_times = []

        for _ in range(N_TIMED):

            start_time = time.perf_counter()

            output = model(x_single)

            if device.type == "cuda":
                torch.cuda.synchronize()

            latency_times.append(
                (time.perf_counter() - start_time) * 1000
            )

            del output

    latency_mean = float(np.mean(latency_times))
    latency_std = float(np.std(latency_times))

    # Throughput measurement
    x_batch = torch.randn(
        BATCH_SIZE_THROUGHPUT,
        in_ch,
        PATCH_SIZE,
        PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        start_time = time.perf_counter()

        for _ in range(N_TIMED):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

    throughput = (
        BATCH_SIZE_THROUGHPUT * N_TIMED
    ) / elapsed_time

    del x_single, x_batch, output
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return latency_mean, latency_std, throughput


# ------------------------------------------------------------
# Independent GPU warm-up
# ------------------------------------------------------------

original_benchmark = torch.backends.cudnn.benchmark
torch.backends.cudnn.benchmark = False

try:

    print("Warming up the GPU...")

    warmup_model = build_profiling_model({
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    }).to(device)

    warmup_input = torch.randn(
        BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    warmup_model.eval()

    with torch.no_grad():
        for _ in range(10):
            warmup_output = warmup_model(warmup_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    del warmup_model, warmup_input, warmup_output

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU warm-up completed.")
    print(f"Profiling device: {device}")

    # --------------------------------------------------------
    # Profiling all five architectures
    # --------------------------------------------------------

    loss_fn_profile = ComboDiceTverskyLoss().to(device)

    profile_rows = []

    for model_name, cfg in ARCH_CONFIGS.items():

        print(f"\nProfiling: {model_name}")

        in_ch = cfg["in_ch"]

        model = build_profiling_model(cfg).to(device)

        total_params, trainable_params = count_params(model)

        size_mb = model_size_mb(model)

        macs, flops = measure_flops(model, in_ch)

        train_memory = measure_train_memory(
            model, in_ch, loss_fn_profile
        )

        inference_memory = measure_inference_memory(
            model, in_ch
        )

        latency_mean, latency_std, throughput = (
            measure_inference_speed(model, in_ch)
        )

        profile_rows.append({
            "Model": model_name,
            "Input": "2D (1-channel)" if in_ch == 1 else "2.5D (3-channel)",
            "Parameters (M)": total_params / 1e6,
            "Trainable Parameters (M)": trainable_params / 1e6,
            "Model Size (MB)": size_mb,
            "MACs (G)": macs / 1e9 if np.isfinite(macs) else np.nan,
            "FLOPs (G)": flops / 1e9 if np.isfinite(flops) else np.nan,
            "Training Memory (MB)": train_memory,
            "Inference Memory (MB)": inference_memory,
            "Latency Mean (ms)": latency_mean,
            "Latency SD (ms)": latency_std,
            "Throughput (images/s)": throughput
        })

        print(
            f"Parameters: {total_params / 1e6:.3f} M | "
            f"Model size: {size_mb:.2f} MB | "
            f"MACs: {macs / 1e9:.2f} G | "
            f"Training memory: {train_memory:.1f} MB | "
            f"Inference memory: {inference_memory:.1f} MB | "
            f"Latency: {latency_mean:.2f} ± {latency_std:.2f} ms | "
            f"Throughput: {throughput:.1f} images/s"
        )

        del model
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Results table
    # --------------------------------------------------------

    df_profile = pd.DataFrame(profile_rows)

    print("\n" + "=" * 110)
    print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
    print("=" * 110)

    display(df_profile.round(3))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    df_profile.to_csv(
        OUTPUT_DIR / "2D_Baseline_Complexity_Full_Table.csv",
        index=False
    )

finally:

    torch.backends.cudnn.benchmark = original_benchmark

    if "loss_fn_profile" in locals():
        del loss_fn_profile

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


Warming up the GPU...
GPU warm-up completed.
Profiling device: cuda

Profiling: 2D_Baseline
Parameters: 34.293 M | Model size: 132.98 MB | MACs: 15.80 G | Training memory: 4400.7 MB | Inference memory: 1129.7 MB | Latency: 16.50 ± 1.87 ms | Throughput: 166.8 images/s

SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Input,Parameters (M),Trainable Parameters (M),Model Size (MB),MACs (G),FLOPs (G),Training Memory (MB),Inference Memory (MB),Latency Mean (ms),Latency SD (ms),Throughput (images/s)
0,2D_Baseline,2D (1-channel),34.293,34.293,132.978,15.8,31.6,4400.739,1129.695,16.501,1.872,166.802


------------

#### 1.2. 2.5D_Baseline Complexity

In [ ]:

# ============================================================
# Segmentation Computational Complexity Analysis
# Five Ablation Architectures
#
# Metrics:
# Parameters, Model Size, MACs, FLOPs,
# Training Memory, Inference Memory,
# Latency, and Throughput
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Warning: thop is not installed. MACs and FLOPs will be unavailable.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH_SIZE = 320
BATCH_SIZE_THROUGHPUT = BATCH_SIZE

N_WARMUP = 25
N_TIMED = 30

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Architecture configurations
# ------------------------------------------------------------

ARCH_CONFIGS = {
    "2.5D_Baseline": {
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    },
}

# ------------------------------------------------------------
# Model utilities
# ------------------------------------------------------------

def build_profiling_model(cfg):
    return SwinUNetAdaptive_DS(
        in_ch=cfg["in_ch"],
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    )


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def model_size_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    total_bytes += sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )
    return total_bytes / (1024 ** 2)


def unpack_model_output(output, training=True):
    if isinstance(output, (tuple, list)):
        main_output = output[0]
        aux2 = output[1] if len(output) > 1 else None
        aux3 = output[2] if len(output) > 2 else None
        attn_mask = (
            output[3]
            if training and len(output) > 3
            else None
        )
    else:
        main_output = output
        aux2 = aux3 = attn_mask = None

    return main_output, aux2, aux3, attn_mask


# ------------------------------------------------------------
# MACs and FLOPs
# ------------------------------------------------------------

def measure_flops(model, in_ch):

    if not HAS_THOP:
        return np.nan, np.nan

    dummy = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    try:
        model.eval()
        macs, _ = thop_profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
        return macs, macs * 2

    except Exception as error:
        print(f"Warning: MACs/FLOPs calculation failed: {error}")
        return np.nan, np.nan

    finally:
        del dummy


# ------------------------------------------------------------
# Training memory
# ------------------------------------------------------------

def measure_train_memory(model, in_ch, loss_fn):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.train()

    x = torch.randn(
        BATCH_SIZE, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    y = torch.randint(
        0, 2,
        (BATCH_SIZE, 1, PATCH_SIZE, PATCH_SIZE),
        device=device
    ).float()

    try:
        output = model(x)

        main_output, aux2, aux3, attn_mask = (
            unpack_model_output(output, training=True)
        )

        loss = loss_fn(
            main_output,
            y,
            aux2=aux2,
            aux3=aux3,
            attn_mask=attn_mask
        )

        loss.backward()
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    finally:
        model.zero_grad(set_to_none=True)

        for variable in [
            x, y, output, main_output,
            aux2, aux3, attn_mask, loss
        ]:
            if variable is not None:
                del variable

        gc.collect()
        torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Inference memory
# ------------------------------------------------------------

def measure_inference_memory(model, in_ch):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()

    with torch.no_grad():
        x = torch.randn(
            BATCH_SIZE_THROUGHPUT,
            in_ch,
            PATCH_SIZE,
            PATCH_SIZE,
            device=device
        )

        output = model(x)
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    del x, output
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Latency and throughput
# ------------------------------------------------------------

def measure_inference_speed(model, in_ch):

    model.eval()

    x_single = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_single)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        latency_times = []

        for _ in range(N_TIMED):

            start_time = time.perf_counter()

            output = model(x_single)

            if device.type == "cuda":
                torch.cuda.synchronize()

            latency_times.append(
                (time.perf_counter() - start_time) * 1000
            )

            del output

    latency_mean = float(np.mean(latency_times))
    latency_std = float(np.std(latency_times))

    # Throughput measurement
    x_batch = torch.randn(
        BATCH_SIZE_THROUGHPUT,
        in_ch,
        PATCH_SIZE,
        PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        start_time = time.perf_counter()

        for _ in range(N_TIMED):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

    throughput = (
        BATCH_SIZE_THROUGHPUT * N_TIMED
    ) / elapsed_time

    del x_single, x_batch, output
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return latency_mean, latency_std, throughput


# ------------------------------------------------------------
# Independent GPU warm-up
# ------------------------------------------------------------

original_benchmark = torch.backends.cudnn.benchmark
torch.backends.cudnn.benchmark = False

try:

    print("Warming up the GPU...")

    warmup_model = build_profiling_model({
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    }).to(device)

    warmup_input = torch.randn(
        BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    warmup_model.eval()

    with torch.no_grad():
        for _ in range(10):
            warmup_output = warmup_model(warmup_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    del warmup_model, warmup_input, warmup_output

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU warm-up completed.")
    print(f"Profiling device: {device}")

    # --------------------------------------------------------
    # Profiling all five architectures
    # --------------------------------------------------------

    loss_fn_profile = ComboDiceTverskyLoss().to(device)

    profile_rows = []

    for model_name, cfg in ARCH_CONFIGS.items():

        print(f"\nProfiling: {model_name}")

        in_ch = cfg["in_ch"]

        model = build_profiling_model(cfg).to(device)

        total_params, trainable_params = count_params(model)

        size_mb = model_size_mb(model)

        macs, flops = measure_flops(model, in_ch)

        train_memory = measure_train_memory(
            model, in_ch, loss_fn_profile
        )

        inference_memory = measure_inference_memory(
            model, in_ch
        )

        latency_mean, latency_std, throughput = (
            measure_inference_speed(model, in_ch)
        )

        profile_rows.append({
            "Model": model_name,
            "Input": "2D (1-channel)" if in_ch == 1 else "2.5D (3-channel)",
            "Parameters (M)": total_params / 1e6,
            "Trainable Parameters (M)": trainable_params / 1e6,
            "Model Size (MB)": size_mb,
            "MACs (G)": macs / 1e9 if np.isfinite(macs) else np.nan,
            "FLOPs (G)": flops / 1e9 if np.isfinite(flops) else np.nan,
            "Training Memory (MB)": train_memory,
            "Inference Memory (MB)": inference_memory,
            "Latency Mean (ms)": latency_mean,
            "Latency SD (ms)": latency_std,
            "Throughput (images/s)": throughput
        })

        print(
            f"Parameters: {total_params / 1e6:.3f} M | "
            f"Model size: {size_mb:.2f} MB | "
            f"MACs: {macs / 1e9:.2f} G | "
            f"Training memory: {train_memory:.1f} MB | "
            f"Inference memory: {inference_memory:.1f} MB | "
            f"Latency: {latency_mean:.2f} ± {latency_std:.2f} ms | "
            f"Throughput: {throughput:.1f} images/s"
        )

        del model
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Results table
    # --------------------------------------------------------

    df_profile = pd.DataFrame(profile_rows)

    print("\n" + "=" * 110)
    print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
    print("=" * 110)

    display(df_profile.round(3))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    df_profile.to_csv(
        OUTPUT_DIR / "2.5_Baseline_Complexity_Full_Table.csv",
        index=False
    )

finally:

    torch.backends.cudnn.benchmark = original_benchmark

    if "loss_fn_profile" in locals():
        del loss_fn_profile

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


Warming up the GPU...
GPU warm-up completed.
Profiling device: cuda

Profiling: 2.5D_Baseline
Parameters: 34.296 M | Model size: 132.99 MB | MACs: 15.82 G | Training memory: 4424.0 MB | Inference memory: 1154.7 MB | Latency: 15.21 ± 0.62 ms | Throughput: 167.2 images/s

SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Input,Parameters (M),Trainable Parameters (M),Model Size (MB),MACs (G),FLOPs (G),Training Memory (MB),Inference Memory (MB),Latency Mean (ms),Latency SD (ms),Throughput (images/s)
0,2.5D_Baseline,2.5D (3-channel),34.296,34.296,132.99,15.82,31.64,4424.032,1154.707,15.208,0.624,167.188


------------

#### 1.3. 2.5D_EncAtt Complexity

In [ ]:

# ============================================================
# Segmentation Computational Complexity Analysis
# Five Ablation Architectures
#
# Metrics:
# Parameters, Model Size, MACs, FLOPs,
# Training Memory, Inference Memory,
# Latency, and Throughput
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Warning: thop is not installed. MACs and FLOPs will be unavailable.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH_SIZE = 320
BATCH_SIZE_THROUGHPUT = BATCH_SIZE

N_WARMUP = 25
N_TIMED = 30

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Architecture configurations
# ------------------------------------------------------------

ARCH_CONFIGS = {
    "2.5D_EncAtt": {
        "in_ch": 3,
        "enc_att": True,
        "dec_att": False
    },
}

# ------------------------------------------------------------
# Model utilities
# ------------------------------------------------------------

def build_profiling_model(cfg):
    return SwinUNetAdaptive_DS(
        in_ch=cfg["in_ch"],
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    )


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def model_size_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    total_bytes += sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )
    return total_bytes / (1024 ** 2)


def unpack_model_output(output, training=True):
    if isinstance(output, (tuple, list)):
        main_output = output[0]
        aux2 = output[1] if len(output) > 1 else None
        aux3 = output[2] if len(output) > 2 else None
        attn_mask = (
            output[3]
            if training and len(output) > 3
            else None
        )
    else:
        main_output = output
        aux2 = aux3 = attn_mask = None

    return main_output, aux2, aux3, attn_mask


# ------------------------------------------------------------
# MACs and FLOPs
# ------------------------------------------------------------

def measure_flops(model, in_ch):

    if not HAS_THOP:
        return np.nan, np.nan

    dummy = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    try:
        model.eval()
        macs, _ = thop_profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
        return macs, macs * 2

    except Exception as error:
        print(f"Warning: MACs/FLOPs calculation failed: {error}")
        return np.nan, np.nan

    finally:
        del dummy


# ------------------------------------------------------------
# Training memory
# ------------------------------------------------------------

def measure_train_memory(model, in_ch, loss_fn):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.train()

    x = torch.randn(
        BATCH_SIZE, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    y = torch.randint(
        0, 2,
        (BATCH_SIZE, 1, PATCH_SIZE, PATCH_SIZE),
        device=device
    ).float()

    try:
        output = model(x)

        main_output, aux2, aux3, attn_mask = (
            unpack_model_output(output, training=True)
        )

        loss = loss_fn(
            main_output,
            y,
            aux2=aux2,
            aux3=aux3,
            attn_mask=attn_mask
        )

        loss.backward()
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    finally:
        model.zero_grad(set_to_none=True)

        for variable in [
            x, y, output, main_output,
            aux2, aux3, attn_mask, loss
        ]:
            if variable is not None:
                del variable

        gc.collect()
        torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Inference memory
# ------------------------------------------------------------

def measure_inference_memory(model, in_ch):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()

    with torch.no_grad():
        x = torch.randn(
            BATCH_SIZE_THROUGHPUT,
            in_ch,
            PATCH_SIZE,
            PATCH_SIZE,
            device=device
        )

        output = model(x)
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    del x, output
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Latency and throughput
# ------------------------------------------------------------

def measure_inference_speed(model, in_ch):

    model.eval()

    x_single = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_single)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        latency_times = []

        for _ in range(N_TIMED):

            start_time = time.perf_counter()

            output = model(x_single)

            if device.type == "cuda":
                torch.cuda.synchronize()

            latency_times.append(
                (time.perf_counter() - start_time) * 1000
            )

            del output

    latency_mean = float(np.mean(latency_times))
    latency_std = float(np.std(latency_times))

    # Throughput measurement
    x_batch = torch.randn(
        BATCH_SIZE_THROUGHPUT,
        in_ch,
        PATCH_SIZE,
        PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        start_time = time.perf_counter()

        for _ in range(N_TIMED):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

    throughput = (
        BATCH_SIZE_THROUGHPUT * N_TIMED
    ) / elapsed_time

    del x_single, x_batch, output
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return latency_mean, latency_std, throughput


# ------------------------------------------------------------
# Independent GPU warm-up
# ------------------------------------------------------------

original_benchmark = torch.backends.cudnn.benchmark
torch.backends.cudnn.benchmark = False

try:

    print("Warming up the GPU...")

    warmup_model = build_profiling_model({
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    }).to(device)

    warmup_input = torch.randn(
        BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    warmup_model.eval()

    with torch.no_grad():
        for _ in range(10):
            warmup_output = warmup_model(warmup_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    del warmup_model, warmup_input, warmup_output

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU warm-up completed.")
    print(f"Profiling device: {device}")

    # --------------------------------------------------------
    # Profiling all five architectures
    # --------------------------------------------------------

    loss_fn_profile = ComboDiceTverskyLoss().to(device)

    profile_rows = []

    for model_name, cfg in ARCH_CONFIGS.items():

        print(f"\nProfiling: {model_name}")

        in_ch = cfg["in_ch"]

        model = build_profiling_model(cfg).to(device)

        total_params, trainable_params = count_params(model)

        size_mb = model_size_mb(model)

        macs, flops = measure_flops(model, in_ch)

        train_memory = measure_train_memory(
            model, in_ch, loss_fn_profile
        )

        inference_memory = measure_inference_memory(
            model, in_ch
        )

        latency_mean, latency_std, throughput = (
            measure_inference_speed(model, in_ch)
        )

        profile_rows.append({
            "Model": model_name,
            "Input": "2D (1-channel)" if in_ch == 1 else "2.5D (3-channel)",
            "Parameters (M)": total_params / 1e6,
            "Trainable Parameters (M)": trainable_params / 1e6,
            "Model Size (MB)": size_mb,
            "MACs (G)": macs / 1e9 if np.isfinite(macs) else np.nan,
            "FLOPs (G)": flops / 1e9 if np.isfinite(flops) else np.nan,
            "Training Memory (MB)": train_memory,
            "Inference Memory (MB)": inference_memory,
            "Latency Mean (ms)": latency_mean,
            "Latency SD (ms)": latency_std,
            "Throughput (images/s)": throughput
        })

        print(
            f"Parameters: {total_params / 1e6:.3f} M | "
            f"Model size: {size_mb:.2f} MB | "
            f"MACs: {macs / 1e9:.2f} G | "
            f"Training memory: {train_memory:.1f} MB | "
            f"Inference memory: {inference_memory:.1f} MB | "
            f"Latency: {latency_mean:.2f} ± {latency_std:.2f} ms | "
            f"Throughput: {throughput:.1f} images/s"
        )

        del model
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Results table
    # --------------------------------------------------------

    df_profile = pd.DataFrame(profile_rows)

    print("\n" + "=" * 110)
    print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
    print("=" * 110)

    display(df_profile.round(3))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    df_profile.to_csv(
        OUTPUT_DIR / "EncAtt_Complexity_Full_Table.csv",
        index=False
    )

finally:

    torch.backends.cudnn.benchmark = original_benchmark

    if "loss_fn_profile" in locals():
        del loss_fn_profile

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


Warming up the GPU...
GPU warm-up completed.
Profiling device: cuda

Profiling: 2.5D_EncAtt
Parameters: 34.854 M | Model size: 135.12 MB | MACs: 15.89 G | Training memory: 4539.5 MB | Inference memory: 1289.4 MB | Latency: 20.10 ± 4.07 ms | Throughput: 165.1 images/s

SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Input,Parameters (M),Trainable Parameters (M),Model Size (MB),MACs (G),FLOPs (G),Training Memory (MB),Inference Memory (MB),Latency Mean (ms),Latency SD (ms),Throughput (images/s)
0,2.5D_EncAtt,2.5D (3-channel),34.854,34.854,135.118,15.894,31.788,4539.512,1289.389,20.102,4.065,165.095


---------------

#### 1.4. 2.5D_DecAtt Complexity

In [ ]:

# ============================================================
# Segmentation Computational Complexity Analysis
# Five Ablation Architectures
#
# Metrics:
# Parameters, Model Size, MACs, FLOPs,
# Training Memory, Inference Memory,
# Latency, and Throughput
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Warning: thop is not installed. MACs and FLOPs will be unavailable.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH_SIZE = 320
BATCH_SIZE_THROUGHPUT = BATCH_SIZE

N_WARMUP = 25
N_TIMED = 30

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Architecture configurations
# ------------------------------------------------------------

ARCH_CONFIGS = {
    "2.5D_DecAtt": {
        "in_ch": 3,
        "enc_att": False,
        "dec_att": True
    },
}

# ------------------------------------------------------------
# Model utilities
# ------------------------------------------------------------

def build_profiling_model(cfg):
    return SwinUNetAdaptive_DS(
        in_ch=cfg["in_ch"],
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    )


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def model_size_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    total_bytes += sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )
    return total_bytes / (1024 ** 2)


def unpack_model_output(output, training=True):
    if isinstance(output, (tuple, list)):
        main_output = output[0]
        aux2 = output[1] if len(output) > 1 else None
        aux3 = output[2] if len(output) > 2 else None
        attn_mask = (
            output[3]
            if training and len(output) > 3
            else None
        )
    else:
        main_output = output
        aux2 = aux3 = attn_mask = None

    return main_output, aux2, aux3, attn_mask


# ------------------------------------------------------------
# MACs and FLOPs
# ------------------------------------------------------------

def measure_flops(model, in_ch):

    if not HAS_THOP:
        return np.nan, np.nan

    dummy = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    try:
        model.eval()
        macs, _ = thop_profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
        return macs, macs * 2

    except Exception as error:
        print(f"Warning: MACs/FLOPs calculation failed: {error}")
        return np.nan, np.nan

    finally:
        del dummy


# ------------------------------------------------------------
# Training memory
# ------------------------------------------------------------

def measure_train_memory(model, in_ch, loss_fn):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.train()

    x = torch.randn(
        BATCH_SIZE, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    y = torch.randint(
        0, 2,
        (BATCH_SIZE, 1, PATCH_SIZE, PATCH_SIZE),
        device=device
    ).float()

    try:
        output = model(x)

        main_output, aux2, aux3, attn_mask = (
            unpack_model_output(output, training=True)
        )

        loss = loss_fn(
            main_output,
            y,
            aux2=aux2,
            aux3=aux3,
            attn_mask=attn_mask
        )

        loss.backward()
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    finally:
        model.zero_grad(set_to_none=True)

        for variable in [
            x, y, output, main_output,
            aux2, aux3, attn_mask, loss
        ]:
            if variable is not None:
                del variable

        gc.collect()
        torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Inference memory
# ------------------------------------------------------------

def measure_inference_memory(model, in_ch):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()

    with torch.no_grad():
        x = torch.randn(
            BATCH_SIZE_THROUGHPUT,
            in_ch,
            PATCH_SIZE,
            PATCH_SIZE,
            device=device
        )

        output = model(x)
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    del x, output
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Latency and throughput
# ------------------------------------------------------------

def measure_inference_speed(model, in_ch):

    model.eval()

    x_single = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_single)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        latency_times = []

        for _ in range(N_TIMED):

            start_time = time.perf_counter()

            output = model(x_single)

            if device.type == "cuda":
                torch.cuda.synchronize()

            latency_times.append(
                (time.perf_counter() - start_time) * 1000
            )

            del output

    latency_mean = float(np.mean(latency_times))
    latency_std = float(np.std(latency_times))

    # Throughput measurement
    x_batch = torch.randn(
        BATCH_SIZE_THROUGHPUT,
        in_ch,
        PATCH_SIZE,
        PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        start_time = time.perf_counter()

        for _ in range(N_TIMED):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

    throughput = (
        BATCH_SIZE_THROUGHPUT * N_TIMED
    ) / elapsed_time

    del x_single, x_batch, output
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return latency_mean, latency_std, throughput


# ------------------------------------------------------------
# Independent GPU warm-up
# ------------------------------------------------------------

original_benchmark = torch.backends.cudnn.benchmark
torch.backends.cudnn.benchmark = False

try:

    print("Warming up the GPU...")

    warmup_model = build_profiling_model({
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    }).to(device)

    warmup_input = torch.randn(
        BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    warmup_model.eval()

    with torch.no_grad():
        for _ in range(10):
            warmup_output = warmup_model(warmup_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    del warmup_model, warmup_input, warmup_output

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU warm-up completed.")
    print(f"Profiling device: {device}")

    # --------------------------------------------------------
    # Profiling all five architectures
    # --------------------------------------------------------

    loss_fn_profile = ComboDiceTverskyLoss().to(device)

    profile_rows = []

    for model_name, cfg in ARCH_CONFIGS.items():

        print(f"\nProfiling: {model_name}")

        in_ch = cfg["in_ch"]

        model = build_profiling_model(cfg).to(device)

        total_params, trainable_params = count_params(model)

        size_mb = model_size_mb(model)

        macs, flops = measure_flops(model, in_ch)

        train_memory = measure_train_memory(
            model, in_ch, loss_fn_profile
        )

        inference_memory = measure_inference_memory(
            model, in_ch
        )

        latency_mean, latency_std, throughput = (
            measure_inference_speed(model, in_ch)
        )

        profile_rows.append({
            "Model": model_name,
            "Input": "2D (1-channel)" if in_ch == 1 else "2.5D (3-channel)",
            "Parameters (M)": total_params / 1e6,
            "Trainable Parameters (M)": trainable_params / 1e6,
            "Model Size (MB)": size_mb,
            "MACs (G)": macs / 1e9 if np.isfinite(macs) else np.nan,
            "FLOPs (G)": flops / 1e9 if np.isfinite(flops) else np.nan,
            "Training Memory (MB)": train_memory,
            "Inference Memory (MB)": inference_memory,
            "Latency Mean (ms)": latency_mean,
            "Latency SD (ms)": latency_std,
            "Throughput (images/s)": throughput
        })

        print(
            f"Parameters: {total_params / 1e6:.3f} M | "
            f"Model size: {size_mb:.2f} MB | "
            f"MACs: {macs / 1e9:.2f} G | "
            f"Training memory: {train_memory:.1f} MB | "
            f"Inference memory: {inference_memory:.1f} MB | "
            f"Latency: {latency_mean:.2f} ± {latency_std:.2f} ms | "
            f"Throughput: {throughput:.1f} images/s"
        )

        del model
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Results table
    # --------------------------------------------------------

    df_profile = pd.DataFrame(profile_rows)

    print("\n" + "=" * 110)
    print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
    print("=" * 110)

    display(df_profile.round(3))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    df_profile.to_csv(
        OUTPUT_DIR / "2.5_DecAtt_Complexity_Full_Table.csv",
        index=False
    )

finally:

    torch.backends.cudnn.benchmark = original_benchmark

    if "loss_fn_profile" in locals():
        del loss_fn_profile

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


Warming up the GPU...
GPU warm-up completed.
Profiling device: cuda

Profiling: 2.5D_DecAtt
Parameters: 34.444 M | Model size: 133.56 MB | MACs: 15.93 G | Training memory: 5053.6 MB | Inference memory: 1754.1 MB | Latency: 18.19 ± 1.33 ms | Throughput: 157.5 images/s

SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Input,Parameters (M),Trainable Parameters (M),Model Size (MB),MACs (G),FLOPs (G),Training Memory (MB),Inference Memory (MB),Latency Mean (ms),Latency SD (ms),Throughput (images/s)
0,2.5D_DecAtt,2.5D (3-channel),34.444,34.444,133.555,15.932,31.865,5053.635,1754.1,18.186,1.335,157.528


--------------------

#### 1.5. 2.5D_FullAtt Complexity

In [ ]:

# ============================================================
# Segmentation Computational Complexity Analysis
# Five Ablation Architectures
#
# Metrics:
# Parameters, Model Size, MACs, FLOPs,
# Training Memory, Inference Memory,
# Latency, and Throughput
# ============================================================

import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Warning: thop is not installed. MACs and FLOPs will be unavailable.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATCH_SIZE = 320
BATCH_SIZE_THROUGHPUT = BATCH_SIZE

N_WARMUP = 25
N_TIMED = 30

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Architecture configurations
# ------------------------------------------------------------

ARCH_CONFIGS = {
    "2.5D_FullAtt": {
        "in_ch": 3,
        "enc_att": True,
        "dec_att": True
    },
}

# ------------------------------------------------------------
# Model utilities
# ------------------------------------------------------------

def build_profiling_model(cfg):
    return SwinUNetAdaptive_DS(
        in_ch=cfg["in_ch"],
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    )


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def model_size_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    total_bytes += sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )
    return total_bytes / (1024 ** 2)


def unpack_model_output(output, training=True):
    if isinstance(output, (tuple, list)):
        main_output = output[0]
        aux2 = output[1] if len(output) > 1 else None
        aux3 = output[2] if len(output) > 2 else None
        attn_mask = (
            output[3]
            if training and len(output) > 3
            else None
        )
    else:
        main_output = output
        aux2 = aux3 = attn_mask = None

    return main_output, aux2, aux3, attn_mask


# ------------------------------------------------------------
# MACs and FLOPs
# ------------------------------------------------------------

def measure_flops(model, in_ch):

    if not HAS_THOP:
        return np.nan, np.nan

    dummy = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    try:
        model.eval()
        macs, _ = thop_profile(
            model,
            inputs=(dummy,),
            verbose=False
        )
        return macs, macs * 2

    except Exception as error:
        print(f"Warning: MACs/FLOPs calculation failed: {error}")
        return np.nan, np.nan

    finally:
        del dummy


# ------------------------------------------------------------
# Training memory
# ------------------------------------------------------------

def measure_train_memory(model, in_ch, loss_fn):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.train()

    x = torch.randn(
        BATCH_SIZE, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    y = torch.randint(
        0, 2,
        (BATCH_SIZE, 1, PATCH_SIZE, PATCH_SIZE),
        device=device
    ).float()

    try:
        output = model(x)

        main_output, aux2, aux3, attn_mask = (
            unpack_model_output(output, training=True)
        )

        loss = loss_fn(
            main_output,
            y,
            aux2=aux2,
            aux3=aux3,
            attn_mask=attn_mask
        )

        loss.backward()
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    finally:
        model.zero_grad(set_to_none=True)

        for variable in [
            x, y, output, main_output,
            aux2, aux3, attn_mask, loss
        ]:
            if variable is not None:
                del variable

        gc.collect()
        torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Inference memory
# ------------------------------------------------------------

def measure_inference_memory(model, in_ch):

    if device.type != "cuda":
        return np.nan

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()

    with torch.no_grad():
        x = torch.randn(
            BATCH_SIZE_THROUGHPUT,
            in_ch,
            PATCH_SIZE,
            PATCH_SIZE,
            device=device
        )

        output = model(x)
        torch.cuda.synchronize()

        peak_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )

    del x, output
    gc.collect()
    torch.cuda.empty_cache()

    return peak_mb


# ------------------------------------------------------------
# Latency and throughput
# ------------------------------------------------------------

def measure_inference_speed(model, in_ch):

    model.eval()

    x_single = torch.randn(
        1, in_ch, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_single)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        latency_times = []

        for _ in range(N_TIMED):

            start_time = time.perf_counter()

            output = model(x_single)

            if device.type == "cuda":
                torch.cuda.synchronize()

            latency_times.append(
                (time.perf_counter() - start_time) * 1000
            )

            del output

    latency_mean = float(np.mean(latency_times))
    latency_std = float(np.std(latency_times))

    # Throughput measurement
    x_batch = torch.randn(
        BATCH_SIZE_THROUGHPUT,
        in_ch,
        PATCH_SIZE,
        PATCH_SIZE,
        device=device
    )

    with torch.no_grad():

        for _ in range(N_WARMUP):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        del output

        start_time = time.perf_counter()

        for _ in range(N_TIMED):
            output = model(x_batch)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.perf_counter() - start_time

    throughput = (
        BATCH_SIZE_THROUGHPUT * N_TIMED
    ) / elapsed_time

    del x_single, x_batch, output
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return latency_mean, latency_std, throughput


# ------------------------------------------------------------
# Independent GPU warm-up
# ------------------------------------------------------------

original_benchmark = torch.backends.cudnn.benchmark
torch.backends.cudnn.benchmark = False

try:

    print("Warming up the GPU...")

    warmup_model = build_profiling_model({
        "in_ch": 3,
        "enc_att": False,
        "dec_att": False
    }).to(device)

    warmup_input = torch.randn(
        BATCH_SIZE, 3, PATCH_SIZE, PATCH_SIZE,
        device=device
    )

    warmup_model.eval()

    with torch.no_grad():
        for _ in range(10):
            warmup_output = warmup_model(warmup_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    del warmup_model, warmup_input, warmup_output

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    print("GPU warm-up completed.")
    print(f"Profiling device: {device}")

    # --------------------------------------------------------
    # Profiling all five architectures
    # --------------------------------------------------------

    loss_fn_profile = ComboDiceTverskyLoss().to(device)

    profile_rows = []

    for model_name, cfg in ARCH_CONFIGS.items():

        print(f"\nProfiling: {model_name}")

        in_ch = cfg["in_ch"]

        model = build_profiling_model(cfg).to(device)

        total_params, trainable_params = count_params(model)

        size_mb = model_size_mb(model)

        macs, flops = measure_flops(model, in_ch)

        train_memory = measure_train_memory(
            model, in_ch, loss_fn_profile
        )

        inference_memory = measure_inference_memory(
            model, in_ch
        )

        latency_mean, latency_std, throughput = (
            measure_inference_speed(model, in_ch)
        )

        profile_rows.append({
            "Model": model_name,
            "Input": "2D (1-channel)" if in_ch == 1 else "2.5D (3-channel)",
            "Parameters (M)": total_params / 1e6,
            "Trainable Parameters (M)": trainable_params / 1e6,
            "Model Size (MB)": size_mb,
            "MACs (G)": macs / 1e9 if np.isfinite(macs) else np.nan,
            "FLOPs (G)": flops / 1e9 if np.isfinite(flops) else np.nan,
            "Training Memory (MB)": train_memory,
            "Inference Memory (MB)": inference_memory,
            "Latency Mean (ms)": latency_mean,
            "Latency SD (ms)": latency_std,
            "Throughput (images/s)": throughput
        })

        print(
            f"Parameters: {total_params / 1e6:.3f} M | "
            f"Model size: {size_mb:.2f} MB | "
            f"MACs: {macs / 1e9:.2f} G | "
            f"Training memory: {train_memory:.1f} MB | "
            f"Inference memory: {inference_memory:.1f} MB | "
            f"Latency: {latency_mean:.2f} ± {latency_std:.2f} ms | "
            f"Throughput: {throughput:.1f} images/s"
        )

        del model
        gc.collect()

        if device.type == "cuda":
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Results table
    # --------------------------------------------------------

    df_profile = pd.DataFrame(profile_rows)

    print("\n" + "=" * 110)
    print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
    print("=" * 110)

    display(df_profile.round(3))

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    df_profile.to_csv(
        OUTPUT_DIR / "2.5D_FullAtt_Complexity_Full_Table.csv",
        index=False
    )

finally:

    torch.backends.cudnn.benchmark = original_benchmark

    if "loss_fn_profile" in locals():
        del loss_fn_profile

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()


Warming up the GPU...
GPU warm-up completed.
Profiling device: cuda

Profiling: 2.5D_FullAtt
Parameters: 35.002 M | Model size: 135.68 MB | MACs: 16.01 G | Training memory: 5169.3 MB | Inference memory: 1888.7 MB | Latency: 19.40 ± 2.04 ms | Throughput: 156.1 images/s

SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Input,Parameters (M),Trainable Parameters (M),Model Size (MB),MACs (G),FLOPs (G),Training Memory (MB),Inference Memory (MB),Latency Mean (ms),Latency SD (ms),Throughput (images/s)
0,2.5D_FullAtt,2.5D (3-channel),35.002,35.002,135.684,16.006,32.013,5169.317,1888.704,19.401,2.037,156.121


-------------------

#### 1.6. Segmentation Complexity Comparison

In [ ]:

# ============================================================
# Combined Segmentation Computational Complexity Table
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RESULTS_DIR = Path("Results/Computational_Complexity_Results/CSV")

MODEL_FILES = {
    "2D_Baseline": "2D_Baseline_Complexity_Full_Table.csv",
    "2.5D_Baseline": "2.5_Baseline_Complexity_Full_Table.csv",
    "2.5D_EncAtt": "2.5D_EncAtt_Complexity_Full_Table.csv",
    "2.5D_DecAtt": "2.5_DecAtt_Complexity_Full_Table.csv",
    "2.5D_FullAtt": "2.5D_FullAtt_Complexity_Full_Table.csv"
}

TABLE_COLUMNS = [
    "Model",
    "Parameters (M)",
    "Model Size (MB)",
    "MACs (G)",
    "Training Memory (MB)",
    "Inference Memory (MB)",
    "Latency (ms)"
]

# ------------------------------------------------------------
# Read and combine results
# ------------------------------------------------------------

all_results = []

for model_name, file_name in MODEL_FILES.items():

    file_path = RESULTS_DIR / file_name

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing complexity result: {file_name}"
        )

    df = pd.read_csv(file_path)

    if df.empty:
        raise ValueError(
            f"The result file is empty: {file_name}"
        )

    row = df.iloc[0]

    latency = (
        f"{float(row['Latency Mean (ms)']):.2f} ± "
        f"{float(row['Latency SD (ms)']):.2f}"
    )

    all_results.append({
        "Model": model_name,
        "Parameters (M)": row["Parameters (M)"],
        "Model Size (MB)": row["Model Size (MB)"],
        "MACs (G)": row["MACs (G)"],
        "Training Memory (MB)": row["Training Memory (MB)"],
        "Inference Memory (MB)": row["Inference Memory (MB)"],
        "Latency (ms)": latency
    })

# ------------------------------------------------------------
# Create final table
# ------------------------------------------------------------

complexity_table = pd.DataFrame(
    all_results,
    columns=TABLE_COLUMNS
)

# ------------------------------------------------------------
# Format numerical columns
# ------------------------------------------------------------

complexity_table["Parameters (M)"] = (
    complexity_table["Parameters (M)"].map(
        lambda x: f"{float(x):.3f}"
    )
)

complexity_table["Model Size (MB)"] = (
    complexity_table["Model Size (MB)"].map(
        lambda x: f"{float(x):.2f}"
    )
)

complexity_table["MACs (G)"] = (
    complexity_table["MACs (G)"].map(
        lambda x: f"{float(x):.2f}"
    )
)

complexity_table["Training Memory (MB)"] = (
    complexity_table["Training Memory (MB)"].map(
        lambda x: f"{float(x):.0f}"
    )
)

complexity_table["Inference Memory (MB)"] = (
    complexity_table["Inference Memory (MB)"].map(
        lambda x: f"{float(x):.0f}"
    )
)

# ------------------------------------------------------------
# Display final table
# ------------------------------------------------------------

print("=" * 100)
print("SEGMENTATION COMPUTATIONAL COMPLEXITY")
print("=" * 100)

display(complexity_table)

# ------------------------------------------------------------
# Save final table
# ------------------------------------------------------------

output_file = (
    RESULTS_DIR / "Segmentation_Complexity_Comparison.csv"
)

complexity_table.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nComplexity comparison table created successfully.")


SEGMENTATION COMPUTATIONAL COMPLEXITY


,Model,Parameters (M),Model Size (MB),MACs (G),Training Memory (MB),Inference Memory (MB),Latency (ms)
0,2D_Baseline,34.293,132.98,15.80,4401,1130,16.50 ± 1.87
1,2.5D_Baseline,34.296,132.99,15.82,4424,1155,15.21 ± 0.62
2,2.5D_EncAtt,34.854,135.12,15.89,4540,1289,20.10 ± 4.07
3,2.5D_DecAtt,34.444,133.56,15.93,5054,1754,18.19 ± 1.33
4,2.5D_FullAtt,35.002,135.68,16.01,5169,1889,19.40 ± 2.04



Complexity comparison table created successfully.


---------------------

### 2. Classification Complexity

#### 2.1. XGBoost Classification Models Complexity

In [ ]:

# ============================================================
# Classification Computational Complexity Analysis
# XGBoost - 25 Trained Models
#
# Metrics:
# Number of Features, Trees, Maximum Depth, Leaf Nodes,
# Model Size, Inference Latency, Throughput,
# and Peak Inference Memory
# ============================================================

import os
import gc
import time
import joblib
import tracemalloc
import numpy as np
import pandas as pd
import xgboost as xgb

from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODEL_SAVE_DIR = Path(
    "Classification/saved_xgb_models_Baseline_FINAL"
)

OUTPUT_DIR = Path(
    "Results/Computational_Complexity_Results/CSV"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_WARMUP = 100
N_TIMED = 1000
BATCH_SIZE_XGB = 256

RANDOM_STATE = 42

# ------------------------------------------------------------
# Locate trained models
# ------------------------------------------------------------

model_files = sorted([
    path for path in MODEL_SAVE_DIR.glob("xgb_seed*.joblib")
])

if len(model_files) == 0:
    raise RuntimeError("No trained XGBoost models were found.")

print("=" * 100)
print("XGBOOST CLASSIFICATION COMPLEXITY ANALYSIS")
print("=" * 100)

print(f"Number of models: {len(model_files)}")

# ------------------------------------------------------------
# Load first bundle and determine feature dimension
# ------------------------------------------------------------

first_bundle = joblib.load(model_files[0])

first_model = first_bundle["model"]
first_scaler = first_bundle["scaler"]

n_features = first_model.n_features_in_

print(f"Number of input features: {n_features}")

del first_bundle, first_model, first_scaler
gc.collect()

# ------------------------------------------------------------
# XGBoost structural complexity
# ------------------------------------------------------------

def count_xgb_structure(model):

    booster = model.get_booster()

    tree_df = booster.trees_to_dataframe()

    n_trees = int(tree_df["Tree"].nunique())

    n_leaves = int(
        (tree_df["Feature"] == "Leaf").sum()
    )

    # Calculate tree depth from node identifiers.
    # XGBoost binary-tree node IDs follow:
    # root = 0, left child = 2n+1, right child = 2n+2.

    tree_depths = []

    for tree_id in tree_df["Tree"].unique():

        tree_nodes = tree_df[
            tree_df["Tree"] == tree_id
        ]

        max_depth = 0

        for node_id in tree_nodes["ID"]:

            try:
                node_number = int(
                    str(node_id).split("-")[-1]
                )

                depth = int(
                    np.floor(np.log2(node_number + 1))
                )

                max_depth = max(max_depth, depth)

            except (ValueError, TypeError):
                continue

        tree_depths.append(max_depth)

    max_depth_overall = (
        max(tree_depths) if tree_depths else np.nan
    )

    return n_trees, n_leaves, max_depth_overall


# ------------------------------------------------------------
# Serialized model bundle size
# ------------------------------------------------------------

def get_model_size_mb(model_path):

    size_bytes = os.path.getsize(model_path)

    return size_bytes / (1024 ** 2)


# ------------------------------------------------------------
# Prepare synthetic input
# ------------------------------------------------------------

def prepare_dummy_input(scaler, n_features):

    rng = np.random.default_rng(RANDOM_STATE)

    X_raw = rng.standard_normal(
        (max(N_TIMED, BATCH_SIZE_XGB), n_features)
    ).astype(np.float32)

    if scaler is not None:
        X_scaled = scaler.transform(X_raw)
        X_scaled = np.asarray(
            X_scaled,
            dtype=np.float32
        )
    else:
        X_scaled = X_raw

    return X_scaled


# ------------------------------------------------------------
# Inference latency
# ------------------------------------------------------------

def measure_xgb_latency(model, X):

    X_warmup = X[:min(N_WARMUP, len(X))]

    for _ in range(N_WARMUP):
        model.predict_proba(X_warmup)

    n_samples = min(N_TIMED, len(X))

    latency_times = []

    for i in range(n_samples):

        x_sample = X[i:i + 1]

        start_time = time.perf_counter()

        model.predict_proba(x_sample)

        elapsed_ms = (
            time.perf_counter() - start_time
        ) * 1000

        latency_times.append(elapsed_ms)

    return (
        float(np.mean(latency_times)),
        float(np.std(latency_times))
    )


# ------------------------------------------------------------
# Throughput
# ------------------------------------------------------------

def measure_xgb_throughput(model, X):

    batch_size = min(BATCH_SIZE_XGB, len(X))

    X_batch = X[:batch_size]

    for _ in range(N_WARMUP):
        model.predict_proba(X_batch)

    n_runs = 100

    start_time = time.perf_counter()

    for _ in range(n_runs):
        model.predict_proba(X_batch)

    elapsed_time = time.perf_counter() - start_time

    throughput = (
        batch_size * n_runs
    ) / elapsed_time

    return float(throughput)


# ------------------------------------------------------------
# Peak Python-tracked inference memory
# ------------------------------------------------------------

def measure_xgb_memory(model, X):

    X_batch = X[:min(BATCH_SIZE_XGB, len(X))]

    gc.collect()

    tracemalloc.start()

    try:
        model.predict_proba(X_batch)

        current_memory, peak_memory = (
            tracemalloc.get_traced_memory()
        )

        peak_mb = peak_memory / (1024 ** 2)

    finally:
        tracemalloc.stop()

    return float(peak_mb)


# ------------------------------------------------------------
# Profile all trained models
# ------------------------------------------------------------

profile_rows = []

print("\nStarting XGBoost profiling...\n")

for index, model_path in enumerate(model_files, start=1):

    print(
        f"Profiling model {index}/{len(model_files)}"
    )

    bundle = joblib.load(model_path)

    model = bundle["model"]
    scaler = bundle["scaler"]

    # Structural complexity
    n_trees, n_leaves, max_depth = (
        count_xgb_structure(model)
    )

    # Serialized bundle size
    size_mb = get_model_size_mb(model_path)

    # Scaled synthetic input
    X_dummy = prepare_dummy_input(
        scaler,
        n_features
    )

    # Inference latency
    latency_mean, latency_std = measure_xgb_latency(
        model,
        X_dummy
    )

    # Throughput
    throughput = measure_xgb_throughput(
        model,
        X_dummy
    )

    # Peak Python-tracked memory
    peak_memory = measure_xgb_memory(
        model,
        X_dummy
    )

    profile_rows.append({
        "Model": "XGBoost",
        "Features": n_features,
        "Trees": n_trees,
        "Max Depth": max_depth,
        "Leaf Nodes": n_leaves,
        "Model Size (MB)": size_mb,
        "Latency Mean (ms)": latency_mean,
        "Latency SD (ms)": latency_std,
        "Throughput (samples/s)": throughput,
        "Peak Inference Memory (MB)": peak_memory
    })

    print(
        f"Features: {n_features} | "
        f"Trees: {n_trees} | "
        f"Leaves: {n_leaves} | "
        f"Max depth: {max_depth} | "
        f"Model size: {size_mb:.3f} MB | "
        f"Latency: {latency_mean:.3f} ± {latency_std:.3f} ms | "
        f"Throughput: {throughput:.1f} samples/s | "
        f"Peak memory: {peak_memory:.4f} MB"
    )

    del bundle, model, scaler, X_dummy

    gc.collect()


# ------------------------------------------------------------
# Individual model results
# ------------------------------------------------------------

df_xgb_complexity = pd.DataFrame(profile_rows)

print("\n" + "=" * 110)
print("XGBOOST COMPLEXITY - INDIVIDUAL MODEL RESULTS")
print("=" * 110)

display(df_xgb_complexity.round(4))

df_xgb_complexity.to_csv(
    OUTPUT_DIR / "XGBoost_Classification_Complexity_Individual.csv",
    index=False
)


# ------------------------------------------------------------
# Summary across all trained models
# ------------------------------------------------------------

summary_columns = [
    "Features",
    "Trees",
    "Max Depth",
    "Leaf Nodes",
    "Model Size (MB)",
    "Latency Mean (ms)",
    "Latency SD (ms)",
    "Throughput (samples/s)",
    "Peak Inference Memory (MB)"
]

summary_rows = []

for column in summary_columns:

    values = pd.to_numeric(
        df_xgb_complexity[column],
        errors="coerce"
    ).dropna()

    summary_rows.append({
        "Metric": column,
        "Mean": values.mean(),
        "SD": values.std(ddof=1)
    })

df_xgb_summary_numeric = pd.DataFrame(summary_rows)

df_xgb_summary = df_xgb_summary_numeric.copy()

df_xgb_summary["Mean ± SD"] = (
    df_xgb_summary["Mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + df_xgb_summary["SD"].map(lambda x: f"{x:.4f}")
)

print("\n" + "=" * 110)
print("XGBOOST COMPLEXITY - SUMMARY ACROSS ALL MODELS")
print("=" * 110)

display(
    df_xgb_summary[["Metric", "Mean ± SD"]]
)

df_xgb_summary_numeric.to_csv(
    OUTPUT_DIR / "XGBoost_Classification_Complexity_Summary_Numeric.csv",
    index=False
)

df_xgb_summary[["Metric", "Mean ± SD"]].to_csv(
    OUTPUT_DIR / "XGBoost_Classification_Complexity_Summary.csv",
    index=False
)

print("\nXGBoost complexity profiling completed.")


XGBOOST CLASSIFICATION COMPLEXITY ANALYSIS
Number of models: 25
Number of input features: 776

Starting XGBoost profiling...

Profiling model 1/25
Features: 776 | Trees: 194 | Leaves: 3012 | Max depth: 4 | Model size: 0.344 MB | Latency: 0.623 ± 0.060 ms | Throughput: 141214.4 samples/s | Peak memory: 0.0141 MB
Profiling model 2/25
Features: 776 | Trees: 236 | Leaves: 3647 | Max depth: 4 | Model size: 0.410 MB | Latency: 0.684 ± 0.148 ms | Throughput: 129913.9 samples/s | Peak memory: 0.0141 MB
Profiling model 3/25
Features: 776 | Trees: 169 | Leaves: 2625 | Max depth: 4 | Model size: 0.303 MB | Latency: 0.670 ± 0.139 ms | Throughput: 136030.9 samples/s | Peak memory: 0.0141 MB
Profiling model 4/25
Features: 776 | Trees: 153 | Leaves: 2399 | Max depth: 4 | Model size: 0.279 MB | Latency: 0.615 ± 0.108 ms | Throughput: 146736.5 samples/s | Peak memory: 0.0141 MB
Profiling model 5/25
Features: 776 | Trees: 148 | Leaves: 2307 | Max depth: 4 | Model size: 0.270 MB | Latency: 0.704 ± 0.121 

,Model,Features,Trees,Max Depth,Leaf Nodes,Model Size (MB),Latency Mean (ms),Latency SD (ms),Throughput (samples/s),Peak Inference Memory (MB)
0,XGBoost,776,194,4,3012,0.3435,0.6228,0.0602,141214.4220,0.0141
1,XGBoost,776,236,4,3647,0.4104,0.6845,0.1484,129913.8915,0.0141
2,XGBoost,776,169,4,2625,0.3031,0.6698,0.1385,136030.9258,0.0141
3,XGBoost,776,153,4,2399,0.2787,0.6153,0.1080,146736.4888,0.0141
4,XGBoost,776,148,4,2307,0.2696,0.7040,0.1213,148160.1519,0.0141
5,XGBoost,776,168,4,2611,0.3016,0.7973,0.1373,150847.4267,0.0141
6,XGBoost,776,145,4,2275,0.2657,0.7881,0.0606,157439.5411,0.0141
7,XGBoost,776,153,4,2400,0.2787,0.7826,0.0492,157799.8752,0.0141
8,XGBoost,776,171,4,2667,0.3071,0.6179,0.0928,152411.8884,0.0141
9,XGBoost,776,175,4,2718,0.3128,0.5671,0.0290,172039.8891,0.0141



XGBOOST COMPLEXITY - SUMMARY ACROSS ALL MODELS


,Metric,Mean ± SD
0,Features,776.0000 ± 0.0000
1,Trees,186.0800 ± 30.2296
2,Max Depth,4.0000 ± 0.0000
3,Leaf Nodes,2887.8000 ± 454.1401
4,Model Size (MB),0.3306 ± 0.0480
5,Latency Mean (ms),0.6798 ± 0.0835
6,Latency SD (ms),0.0898 ± 0.0470
7,Throughput (samples/s),148751.8740 ± 10997.3194
8,Peak Inference Memory (MB),0.0141 ± 0.0000



XGBoost complexity profiling completed.


------------------

### 3. Overall Framework Complexity

In [ ]:

# ============================================================
# Overall Framework Computational Complexity
# Classification + Segmentation + Complete Framework
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RESULTS_DIR = Path("Results/Computational_Complexity_Results/CSV")

SEGMENTATION_FILE = (
    RESULTS_DIR / "Segmentation_Complexity_Comparison.csv"
)

OUTPUT_FILE = (
    RESULTS_DIR / "Overall_Framework_Complexity.csv"
)

# ------------------------------------------------------------
# Load segmentation results
# ------------------------------------------------------------

if not SEGMENTATION_FILE.exists():
    raise FileNotFoundError(
        "The segmentation complexity comparison file was not found."
    )

df_seg = pd.read_csv(SEGMENTATION_FILE)

decatt_row = df_seg.loc[
    df_seg["Model"] == "2.5D_DecAtt"
]

if decatt_row.empty:
    raise ValueError(
        "The 2.5D_DecAtt results were not found."
    )

decatt = decatt_row.iloc[0]

seg_model_size = float(decatt["Model Size (MB)"])
seg_macs = float(decatt["MACs (G)"])
seg_inference_memory = float(decatt["Inference Memory (MB)"])

# Extract latency mean if stored as "Mean ± SD"
seg_latency_value = str(decatt["Latency (ms)"])
seg_latency = float(seg_latency_value.split("±")[0].strip())

# ------------------------------------------------------------
# Classification results
# Mean values across 25 XGBoost models
# ------------------------------------------------------------

cls_model_size = 0.3306
cls_inference_memory = 0.0141
cls_latency = 0.6798

# XGBoost MACs are not reported
cls_macs = "—"

# ------------------------------------------------------------
# Complete framework
# ------------------------------------------------------------

framework_model_size = cls_model_size + seg_model_size

# CPU and GPU memory are reported separately
framework_inference_memory = "Separate"

# End-to-end latency has not been measured
framework_latency = "—"

framework_macs = seg_macs

# ------------------------------------------------------------
# Build final table
# ------------------------------------------------------------

complexity_data = [
    {
        "Stage": "Classification",
        "Model Size (MB)": round(cls_model_size, 2),
        "MACs (G)": cls_macs,
        "Inference Mem. (MB)": round(cls_inference_memory, 3),
        "Latency (ms)": round(cls_latency, 2),
        "Hardware": "CPU"
    },
    {
        "Stage": "Segmentation",
        "Model Size (MB)": round(seg_model_size, 2),
        "MACs (G)": round(seg_macs, 2),
        "Inference Mem. (MB)": round(seg_inference_memory, 2),
        "Latency (ms)": round(seg_latency, 2),
        "Hardware": "GPU"
    },
    {
        "Stage": "Complete framework",
        "Model Size (MB)": round(framework_model_size, 2),
        "MACs (G)": round(framework_macs, 2),
        "Inference Mem. (MB)": framework_inference_memory,
        "Latency (ms)": framework_latency,
        "Hardware": "CPU + GPU"
    }
]

complexity_table = pd.DataFrame(complexity_data)

# ------------------------------------------------------------
# Display final table
# ------------------------------------------------------------

print("=" * 100)
print("OVERALL FRAMEWORK COMPUTATIONAL COMPLEXITY")
print("=" * 100)

display(complexity_table)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

complexity_table.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nOverall framework complexity table created successfully.")


OVERALL FRAMEWORK COMPUTATIONAL COMPLEXITY


,Stage,Model Size (MB),MACs (G),Inference Mem. (MB),Latency (ms),Hardware
0,Classification,0.33,—,0.014,0.68,CPU
1,Segmentation,133.56,15.93,1754.0,18.19,GPU
2,Complete framework,133.89,15.93,Separate,—,CPU + GPU



Overall framework complexity table created successfully.
